In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lakefs_spec
import tensorflow_privacy as tpf
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow_privacy.privacy.analysis import compute_dp_sgd_privacy_lib

In [2]:
# Load data using lakefs
LAKEFS_REPO = "athletes"
lakefs_uri = lakefs_uri = f"lakefs://{LAKEFS_REPO}/v2/athletes_v2.csv"

# Open file using lakefs
fs = lakefs_spec.LakeFSFileSystem()

with fs.open(lakefs_uri, "rb") as f:
    df = pd.read_csv(f)

# Load data using DVC after git checkout v2 and dvc checkout
# df = pd.read_csv('athletes.csv')

print(f"Loaded dataset v2.")

Loaded dataset v2.


## Preprocessing
(Similar to the previous model)

In [3]:
# Total lift
df['total_lift'] = df['deadlift'] + df['candj'] + df['snatch'] + df['backsq']

# Train test split
df_train, df_test = train_test_split(df, test_size=0.2, random_state=32021)

In [4]:
df_train = df_train.dropna(subset=['total_lift'])
df_test = df_test.dropna(subset=['total_lift'])

target = 'total_lift'

X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]

In [5]:
# Numeric and categorical predictors
num_vars = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_vars = ['gender','howlong']

In [6]:
# Keep only relevant predictors
X_train = X_train[num_vars + cat_vars]
X_test = X_test[num_vars + cat_vars]

num_imputer = SimpleImputer(strategy='median')
X_train[num_vars] = num_imputer.fit_transform(X_train[num_vars])
X_test[num_vars] = num_imputer.transform(X_test[num_vars])

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[cat_vars] = cat_imputer.fit_transform(X_train[cat_vars])
X_test[cat_vars] = cat_imputer.transform(X_test[cat_vars])

In [7]:
# Encode categorical
X_train = pd.get_dummies(X_train, columns=cat_vars)
X_test = pd.get_dummies(X_test, columns=cat_vars)


scaler = StandardScaler()
X_train[num_vars] = scaler.fit_transform(X_train[num_vars])
X_test[num_vars] = scaler.transform(X_test[num_vars])


X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [8]:
X_train = tf.convert_to_tensor(X_train.values, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train.values, dtype=tf.float32)

X_test = tf.convert_to_tensor(X_test.values, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test.values, dtype=tf.float32)

## DP Model

In [9]:
# Build Keras model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(1)
])

batch_size = 32 
epochs = 20
noise_multiplier = 0.8
optimizer = DPKerasSGDOptimizer(
    l2_norm_clip=0.8,
    noise_multiplier=noise_multiplier,
    num_microbatches=1,
    learning_rate=0.05
)

model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

In [10]:
model_train = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=1
)

Epoch 1/20
679/679 [==============================] - 1s 422us/step - loss: 827339.1875 - mae: 870.2991 - val_loss: 363281.9688 - val_mae: 584.5216
Epoch 2/20
679/679 [==============================] - 0s 337us/step - loss: 97392.2422 - mae: 230.8086 - val_loss: 1135.0394 - val_mae: 26.2119
Epoch 3/20
679/679 [==============================] - 0s 335us/step - loss: 3938.1194 - mae: 49.0200 - val_loss: 1012.8926 - val_mae: 24.0765
Epoch 4/20
679/679 [==============================] - 0s 334us/step - loss: 3897.9116 - mae: 48.7614 - val_loss: 824.8422 - val_mae: 22.2649
Epoch 5/20
679/679 [==============================] - 0s 394us/step - loss: 4140.9653 - mae: 49.8358 - val_loss: 1209.1252 - val_mae: 27.5567
Epoch 6/20
679/679 [==============================] - 0s 335us/step - loss: 4018.9465 - mae: 49.2055 - val_loss: 690.3480 - val_mae: 19.7754
Epoch 7/20
679/679 [==============================] - 0s 345us/step - loss: 3876.3174 - mae: 48.5240 - val_loss: 1549.3037 - val_mae: 28.9479


In [11]:
y_pred = model.predict(X_test).flatten()
y_test = y_test.numpy()

# Calculate metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Metrics:")
print(f"R^2 score: {r2}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")

189/189 [==============================] - 0s 200us/step
Metrics:
R^2 score: 0.9881852269172668
MAE: 22.680789947509766
MSE: 897.1331787109375
RMSE: 29.952181535089185


In [12]:
# Compute DP epsilon
epsilon, _ = compute_dp_sgd_privacy_lib.compute_dp_sgd_privacy(
    n=len(X_train),
    batch_size=batch_size,
    noise_multiplier=noise_multiplier,
    epochs=epochs,
    delta=1e-5
)

print(f"DP Model's epsilon: {epsilon:.4f}")

DP Model's epsilon: 1.8192
